# EDA 10: District crop production (APY) — load, tidy, map to LGD

**Source:** `apy_idp`, India Data Portal (ISB) mirror of DES district Area-Production-Yield,
two resources: **DES** (1997-98 to 2022-23, more crops) and **UPAG** (2013-14 to 2024-25,
field crops). Licence ODC-By 1.0 (see `docs/data-sources.md`).

**Question:** Can district crop production for the ethanol feedstocks (sugarcane; maize and
rice for grain ethanol) and for CBG residue (+ wheat) reach all 75 districts by LGD code,
for recent years, in consistent units?

| | |
|---|---|
| **Inputs** | the IDP CKAN datastore API (UP rows only), or the raw CSVs already in `data/raw/apy_idp/ingest_date=*/` |
| **Outputs** | `data/interim/apy/apy_up_long.parquet`: one row per (resource, crop_year, season, crop, lgd_district_code) with `area_ha`, `production_t`, `yield_t_per_ha` |
| **Grain** | district × crop × season × year (its own grain; the district table is Sage's EDA 03) |
| **Run** | *Kernel → Restart and Run All*. Fetches only if no raw files exist yet; set `UP_SITING_APY_REFETCH=1` to fetch a new snapshot |

Why the API and not the 56 MB CSV: the CSV download redirects to a storage host this
project's cloud session can't reach, and we only need UP. The datastore API serves the
same rows (same resource ids), filtered to `state_name = 'Uttar Pradesh'`.

In [1]:
import json
import os
import urllib.parse
import urllib.request
from datetime import datetime
from zoneinfo import ZoneInfo

import duckdb
import pandas as pd

from up_siting.common.names import map_to_lgd
from up_siting.common.paths import INTERIM_DIR, RAW_DIR, SEEDS_DIR
from up_siting.extract.manifest import register_file

SOURCE = "apy_idp"
API = "https://ckandev.indiadataportal.com/api/3/action/"
DATASET_PAGE = "https://ckandev.indiadataportal.com/dataset/area-production-yield-apy"
RESOURCES = {  # our short name -> CKAN resource id
    "des": "f980409d-49a2-42ae-9eb0-182365005c04",  # Crop Wise Area Production Yield (DES)
    "upag": "6f70d961-3a4f-4caa-b08f-f72f708791e1",  # Crop-wise Area, Production & Yield (UPAG)
}
OUT_DIR = INTERIM_DIR / "apy"

spine = pd.read_csv(SEEDS_DIR / "lgd_district_spine.csv")
name_map = pd.read_csv(SEEDS_DIR / "district_name_map.csv")
print(len(spine), "districts in the spine")

75 districts in the spine


## 1. Fetch (once) and register the raw files

Each resource is paged through `datastore_search` with a `state_name` filter and saved
unchanged as CSV, then registered in `data/raw/apy_idp/_manifest.jsonl` with
`fetched_by="extractor"`. A later run reuses the newest snapshot, so the notebook works
offline once the files exist. *(Phase 2 moves this fetch into an Airflow extractor; the
manifest stays the same.)*

In [2]:
def latest_snapshot(source: str, files: list[str]):
    # Newest ingest_date folder that holds every file, or None.
    folders = sorted((RAW_DIR / source).glob("ingest_date=*"), reverse=True)
    for folder in folders:
        if all((folder / f).is_file() for f in files):
            return folder
    return None


def fetch_up_rows(resource_id: str, page_size: int = 10_000) -> pd.DataFrame:
    # Page through the CKAN datastore, UP rows only; returns everything as text.
    frames, offset = [], 0
    filters = json.dumps({"state_name": "Uttar Pradesh"})
    while True:
        query = urllib.parse.urlencode(
            {"resource_id": resource_id, "filters": filters, "limit": page_size, "offset": offset}
        )
        with urllib.request.urlopen(API + "datastore_search?" + query, timeout=120) as resp:
            result = json.load(resp)["result"]
        records = result["records"]
        if not records:
            break
        frames.append(pd.DataFrame.from_records(records))
        offset += len(records)
        if offset >= result["total"]:
            break
    df = pd.concat(frames, ignore_index=True)
    assert len(df) == result["total"], (len(df), result["total"])
    return df


FILES = {name: f"apy_{name}_up.csv" for name in RESOURCES}
snapshot = latest_snapshot(SOURCE, list(FILES.values()))
if snapshot is None or os.environ.get("UP_SITING_APY_REFETCH") == "1":
    today = datetime.now(ZoneInfo("Asia/Kolkata")).date().isoformat()
    snapshot = RAW_DIR / SOURCE / f"ingest_date={today}"
    snapshot.mkdir(parents=True, exist_ok=True)
    for name, rid in RESOURCES.items():
        path = snapshot / FILES[name]
        fetch_up_rows(rid).to_csv(path, index=False)
        register_file(
            SOURCE,
            path,
            f'{API}datastore_search?resource_id={rid}&filters={{"state_name":"Uttar Pradesh"}}',
            fetched_by="extractor",
            notes=f"{name} resource of {DATASET_PAGE}; UP rows only via the CKAN datastore API",
        )
print("using", snapshot.relative_to(RAW_DIR.parent.parent))

using data/raw/apy_idp/ingest_date=2026-11-24


In [3]:
manifest = duckdb.sql(
    f"select file, sha256[:12] as sha, bytes, fetched_at, licence "
    f"from read_json('{(RAW_DIR / SOURCE / '_manifest.jsonl').as_posix()}')"
).df()
manifest

,file,sha,bytes,fetched_at,licence
0,data/raw/apy_idp/ingest_date=2026-11-24/apy_de...,bb23519c8b9b,7404684,2026-11-24 14:39:12+00:00,ODC-By 1.0 (Open Data Commons Attribution; CKA...
1,data/raw/apy_idp/ingest_date=2026-11-24/apy_up...,7c48bf4d60f7,1932149,2026-11-24 14:39:17+00:00,ODC-By 1.0 (Open Data Commons Attribution; CKA...


In [4]:
assert len(manifest) >= 2, "both resources must be registered"
assert manifest["licence"].str.contains("ODC-By").all()

## 2. What does each resource contain?

Read both CSVs as text (as in EDA 01: cast later, on purpose) and profile them with DuckDB.

In [5]:
raw = {
    name: pd.read_csv(snapshot / file, dtype=str, keep_default_na=False)
    for name, file in FILES.items()
}
for name, df in raw.items():
    print(name, df.shape, list(df.columns))

des (56511, 17) ['_id', 'id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'crop_name', 'crop_code', 'crop_type', 'season', 'area', 'area_unit', 'production', 'production_unit', 'yield', 'yield_unit']
upag (19208, 14) ['_id', 'id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'season', 'crop_category', 'crop_name', 'crop_code', 'area', 'production', 'yield']


In [6]:
des, upag = raw["des"], raw["upag"]
profile = duckdb.sql("""
    select 'des' as res, count(*) as n_rows, min(year) as first_year, max(year) as last_year,
           count(distinct district_code) as n_districts, count(distinct crop_name) as n_crops,
           string_agg(distinct season, ', ' order by season) as seasons
    from des
    union all
    select 'upag', count(*), min(year), max(year), count(distinct district_code),
           count(distinct crop_name), string_agg(distinct season, ', ' order by season)
    from upag
""").df()
profile

,res,n_rows,first_year,last_year,n_districts,n_crops,seasons
0,des,56511,1997-1998,2022-2023,75,39,"Kharif, Rabi, Summer, Total, Whole Year"
1,upag,19208,2013-2014,2024-2025,75,24,"Kharif, Rabi, Summer"


In [7]:
# Units: DES states them per row; UPAG doesn't (the IDP page says ha, tonnes, kg/ha).
units = duckdb.sql(
    "select area_unit, production_unit, yield_unit, count(*) n from des group by all"
).df()
print(units)
assert len(units) == 1 and tuple(units.iloc[0, :3]) == ("Hectare", "Tonnes", "Tonnes/Hectare")

  area_unit production_unit      yield_unit      n
0   Hectare          Tonnes  Tonnes/Hectare  56511


## 3. Is `district_code` the LGD district code?

The mirror says it is "LGD-mapped". Check it two ways: every code is in the spine, and the
code agrees with what our own name map (EDA 01) says for the printed name.

In [8]:
checks = []
for name, df in raw.items():
    pairs = df[["district_name", "district_code"]].drop_duplicates()
    # map_to_lgd is Sage's function; pass a copy of name_map (see TODO-sage review note).
    mapped, unmatched = map_to_lgd(pairs, "district_name", name_map.copy())
    mapped["code_from_source"] = mapped["district_code"].astype("Int64")
    mapped["agree"] = mapped["code_from_source"] == mapped["lgd_district_code"]
    checks.append(mapped.assign(resource=name))
    print(name, "unmatched names:", unmatched.to_dict("records"))
code_check = pd.concat(checks, ignore_index=True)
code_check[~code_check["agree"].fillna(False)]

des unmatched names: []
upag unmatched names: []


,district_name,district_code,lgd_district_code,code_from_source,agree,resource


In [9]:
spine_codes = set(spine["lgd_district_code"])
for name, df in raw.items():
    codes = set(df["district_code"].astype(int))
    assert codes == spine_codes, (name, codes ^ spine_codes)
assert code_check["agree"].all(), "a district code disagrees with the name map"
assert code_check.groupby("resource")["district_code"].nunique().eq(75).all()

## 4. Tidy: one long table, one unit system

* Crop names differ between resources ("Arhar/Tur" vs "Tur", "Moong(Green Gram)" vs
  "Moong"): a small map gives a common `crop` id for the crops the profiles use.
* DES has **`Total` season rows** (Maize, Rice, Urad, Onion, …) that are the sum of the
  seasons: keeping them would double-count. They're dropped; `Whole Year` rows (sugarcane,
  potato, …) are real and kept.
* UPAG yield is kg/ha; DES yield is t/ha. Both become `yield_t_per_ha`, and both are
  recomputed from production ÷ area as a check.
* `crop_year` = the first calendar year of the agricultural year ("2022-2023" → 2022).

In [10]:
CROP_IDS = {  # crops the ethanol and CBG profiles use; others keep a snake_case id
    "Sugarcane": "sugarcane",
    "Rice": "rice",
    "Maize": "maize",
    "Wheat": "wheat",
    "Bajra": "bajra",
    "Jowar": "jowar",
    "Barley": "barley",
}


def crop_id(name: str) -> str:
    if name in CROP_IDS:
        return CROP_IDS[name]
    return "".join(c if c.isalnum() else "_" for c in name.lower()).strip("_").replace("__", "_")


tidy = duckdb.sql("""
    with des_rows as (
        select 'des' as resource, year, district_code, crop_name, season,
               cast(nullif(area, '') as double) as area_ha,
               cast(nullif(production, '') as double) as production_t,
               cast(nullif(yield, '') as double) as yield_reported_t_per_ha
        from des
        where season <> 'Total'
    ),
    upag_rows as (
        select 'upag', year, district_code, crop_name, season,
               cast(nullif(area, '') as double), cast(nullif(production, '') as double),
               cast(nullif(yield, '') as double) / 1000.0           -- kg/ha -> t/ha
        from upag
    )
    select resource,
           cast(substr(year, 1, 4) as integer) as crop_year,
           year as crop_year_label,
           cast(district_code as integer) as lgd_district_code,
           crop_name, season, area_ha, production_t, yield_reported_t_per_ha,
           production_t / nullif(area_ha, 0) as yield_t_per_ha
    from (select * from des_rows union all select * from upag_rows)
""").df()
tidy["crop"] = tidy["crop_name"].map(crop_id)
tidy["lgd_district_code"] = tidy["lgd_district_code"].astype("Int64")
print(tidy.shape)
tidy.head()

(69746, 11)


,resource,crop_year,crop_year_label,lgd_district_code,crop_name,season,area_ha,production_t,yield_reported_t_per_ha,yield_t_per_ha,crop
0,des,1997,1997-1998,118,Arhar/Tur,Kharif,6538.0,9664.0,1.478,1.478128,arhar_tur
1,des,1997,1997-1998,118,Bajra,Kharif,99898.0,147994.0,1.481,1.481451,bajra
2,des,1997,1997-1998,118,Barley,Rabi,10865.0,35423.0,3.260,3.260285,barley
3,des,1997,1997-1998,118,Gram,Rabi,11744.0,11816.0,1.006,1.006131,gram
4,des,1997,1997-1998,118,Groundnut,Kharif,27.0,22.0,0.815,0.814815,groundnut


In [11]:
key = ["resource", "crop_year", "season", "crop", "lgd_district_code"]
dupes = tidy[tidy.duplicated(key, keep=False)]
assert dupes.empty, dupes.head()
assert not (tidy["season"] == "Total").any()
# Reported yield vs production / area: they should agree (rounding aside) wherever both exist.
both = tidy.dropna(subset=["yield_reported_t_per_ha", "yield_t_per_ha"])
both = both[both["area_ha"] > 100]
rel_gap = (both["yield_reported_t_per_ha"] / both["yield_t_per_ha"] - 1).abs()
print(f"yield check: {len(both):,} rows; {(rel_gap > 0.02).mean():.2%} differ by more than 2%")
assert (rel_gap > 0.02).mean() < 0.02, "units look inconsistent"

yield check: 46,823 rows; 0.00% differ by more than 2%


## 5. Do the two resources agree where they overlap (2013-14 to 2022-23)?

If UPAG reproduces DES in the overlap, the long table can use **DES before 2013 and UPAG
from 2013** without a break. Compared on annual district totals (all seasons summed).

In [12]:
FOCUS = ["sugarcane", "rice", "maize", "wheat"]
annual = (
    tidy[tidy["crop"].isin(FOCUS)]
    .groupby(["resource", "crop", "crop_year", "lgd_district_code"], as_index=False)[
        ["area_ha", "production_t"]
    ]
    .sum(min_count=1)
)
wide = annual.pivot_table(
    index=["crop", "crop_year", "lgd_district_code"], columns="resource", values="production_t"
).dropna()
wide["ratio"] = wide["upag"] / wide["des"]
agreement = wide.groupby("crop")["ratio"].describe(percentiles=[0.05, 0.5, 0.95])
agreement

,count,mean,std,min,5%,50%,95%,max
crop,,,,,,,,
maize,743.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
rice,750.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
sugarcane,660.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
wheat,750.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0


In [13]:
within_1pct = (wide["ratio"].sub(1).abs() <= 0.01).groupby(level="crop").mean()
print(within_1pct.round(3).to_dict())
assert (within_1pct > 0.9).all(), "UPAG and DES disagree in the overlap years"

{'maize': 1.0, 'rice': 1.0, 'sugarcane': 1.0, 'wheat': 1.0}


## 6. Coverage of the focus crops: years × districts

For each focus crop, how many of the 75 districts report production in each year?

In [14]:
best = annual[(annual["resource"] == "upag") | (annual["crop_year"] < 2013)]
coverage = (
    best[best["production_t"] > 0]
    .groupby(["crop", "crop_year"])["lgd_district_code"]
    .nunique()
    .unstack("crop")
)
coverage.tail(14)

crop,maize,rice,sugarcane,wheat
crop_year,,,,
2011,72,72,72,72
2012,73,75,75,75
2013,75,75,75,75
2014,74,75,75,75
2015,75,75,75,75
2016,73,75,75,75
2017,75,75,75,75
2018,75,75,75,75
2019,75,75,75,75


In [15]:
recent = coverage.loc[2021:2024]
print(recent)
assert (recent["rice"] >= 70).all() and (recent["wheat"] >= 70).all()
assert (recent["maize"] >= 60).all()
# Sugarcane is NOT complete from 2020-21: see section 7.

crop       maize  rice  sugarcane  wheat
crop_year                               
2021          72    75         45     75
2022          74    75         45     75
2023          75    75         44     75
2024          75    75         45     75


## 7. Sugarcane: the 2020-21 break

From 2020-21 sugarcane is reported for only **45** districts, in *both* resources, and the
state total jumps the same year. Which districts drop out, and how much cane did they have
before the break?

In [16]:
cane = annual[(annual["crop"] == "sugarcane") & (annual["resource"] == "des")]
cane_upag = annual[(annual["crop"] == "sugarcane") & (annual["resource"] == "upag")]
latest_year = int(cane_upag["crop_year"].max())
before = (
    cane[cane["crop_year"].between(2017, 2019)].groupby("lgd_district_code")["production_t"].mean()
)
# The set of reporting districts shifts a little from year to year (47 appear at least once
# in 2020-22), so "dropped" means: had cane before the break, absent in the LATEST year.
after = set(cane_upag.loc[cane_upag["crop_year"] == latest_year, "lgd_district_code"])
dropped = (
    before[~before.index.isin(after)]
    .rename("mean_production_t_2017_19")
    .reset_index()
    .merge(spine[["lgd_district_code", "district_name"]], on="lgd_district_code")
    .sort_values("mean_production_t_2017_19", ascending=False)
)
state_cane = cane.groupby("crop_year")["production_t"].agg(["count", "sum"])
state_cane["sum"] = state_cane["sum"] / 1e6
print(state_cane.loc[2016:2022].rename(columns={"count": "n_districts", "sum": "state_mt"}))
print(
    f"{len(dropped)} districts absent in {latest_year}-{latest_year - 1999}; they had "
    f"{dropped['mean_production_t_2017_19'].sum() / 1e6:.1f} Mt/yr in 2017-19 "
    f"({dropped['mean_production_t_2017_19'].sum() / before.sum():.1%} of the state)"
)
dropped.head(12)

           n_districts    state_mt
crop_year                         
2016                75  156.948671
2017                75  177.056410
2018                75  179.698158
2019                75  179.567765
2020                45  223.283000
2021                45  227.220000
2022                45  239.436000
30 districts absent in 2024-25; they had 2.7 Mt/yr in 2017-19 (1.5% of the state)


,lgd_district_code,mean_production_t_2017_19,district_name
26,181,579266.666667,Shrawasti
8,142,333360.333333,Fatehpur
24,178,246069.333333,Sant Kabir Nagar
11,149,177923.333333,Hamirpur
10,144,172789.666667,Gautam Buddha Nagar
15,156,142223.333333,Kanpur Dehat
22,170,133486.666667,Mirzapur
29,640,123376.000000,Amethi
20,165,93539.666667,Mahoba
25,179,83602.333333,Bhadohi


In [17]:
assert len(before) == 75 and len(dropped) + len(after) == 75
assert set(coverage.loc[2020:2024, "sugarcane"]) <= {44, 45}
assert set(coverage.loc[2013:2019, "sugarcane"]) == {75}

## 8. A smell test on 2024-25

UPAG's 2024-25 rows have decimals where earlier years are whole numbers, which suggests a
new estimation method (or provisional figures). How much does the state total move?

In [18]:
state = (
    best.groupby(["crop", "crop_year"])[["area_ha", "production_t"]]
    .sum()
    .div(1e6)
    .rename(columns={"area_ha": "area_mha", "production_t": "production_mt"})
    .unstack("crop")
)
state.loc[2018:2024].round(2)

area_mha                        production_mt                   \
crop         maize  rice sugarcane  wheat         maize   rice sugarcane   
crop_year                                                                  
2018          0.73  5.92      2.22   9.86          1.52  16.02    179.70   
2019          0.73  5.92      2.21   9.85          1.80  17.03    179.57   
2020          0.77  5.97      2.74   9.85          1.82  17.23    223.28   
2021          0.89  6.18      2.76  10.19          1.92  16.55    227.22   
2022          0.89  6.18      2.85  10.19          1.95  16.56    239.44   
2023          1.10  5.77      2.65   9.53          2.67  15.99    215.81   
2024          0.68  7.35      2.72   9.60          1.96  20.76    220.80   

                  
crop       wheat  
crop_year         
2018       38.04  
2019       36.21  
2020       37.48  
2021       38.06  
2022       38.07  
2023       35.34  
2024       35.65

In [19]:
kheri = best[(best["lgd_district_code"] == 159) & best["crop_year"].between(2020, 2024)]
kheri.pivot_table(index="crop_year", columns="crop", values="production_t").round(0)

crop,maize,rice,sugarcane,wheat
crop_year,,,,
2020,19525.0,640169.0,28577000.0,802091.0
2021,18542.0,530375.0,28047000.0,827717.0
2022,17716.0,521373.0,30285000.0,798068.0
2023,37431.0,524878.0,28728557.0,632955.0
2024,13903.0,380261.0,28405598.0,728379.0


## 9. Write the tidy table and read it back

In [20]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out = OUT_DIR / "apy_up_long.parquet"
cols = [
    "resource",
    "crop_year",
    "crop_year_label",
    "lgd_district_code",
    "crop",
    "crop_name",
    "season",
    "area_ha",
    "production_t",
    "yield_t_per_ha",
    "yield_reported_t_per_ha",
]
tidy[cols].to_parquet(out, index=False)
back = duckdb.sql(
    f"select count(*) n, count(distinct lgd_district_code) d from '{out.as_posix()}'"
).df()
back

,n,d
0,69746,75


In [21]:
assert back.loc[0, "n"] == len(tidy) and back.loc[0, "d"] == 75

## 10. Findings

Filled in from the run on 2026-11-24 (numbers are in the outputs above).

1. **All 75 districts, LGD-keyed.** Both resources carry the LGD district code, and it agrees
   with our name map for every district (§3). No aliases needed.
2. **Years.** DES 1997-98 to 2022-23; UPAG 2013-14 to **2024-25**. In the overlap, UPAG
   reproduces DES for sugarcane, rice, maize and wheat (§5), so the series can switch to
   UPAG from 2013 without a break.
3. **Double counting trap.** DES has `Total` season rows (sum of the seasons) for maize,
   rice and others. Dropped in §4. → DQ-005.
4. **Units.** Area ha, production tonnes; yield t/ha (DES) vs kg/ha (UPAG), standardised to
   t/ha and checked against production ÷ area.
5. **2024-25 looks different** (§7): fractional values and large drops in some districts
   (Kheri rice production falls from ≈0.52 to 0.38 Mt while the state rice total rises
   from 16.0 to 20.8 Mt; maize spikes to 2.7 Mt in 2023-24 then falls back). Treat 2024-25 as provisional / a possible method
   change until DES's own portal confirms it. → DQ-006.
6. **Sugarcane has a break in 2020-21** (§7): from that year only 45 districts report cane
   (both resources), and the state total jumps (≈180 → 223 Mt; cane *area* jumps too,
   2.2 → 2.7 Mha). The 30 districts absent in 2024-25 had 2.7 Mt/yr in 2017-19 (1.5% of
   the state), but some are not trivial (Shrawasti 0.58 Mt, Fatehpur 0.33 Mt, Sant Kabir
   Nagar 0.25 Mt). Likely a switch to the cane department's survey of mill command
   areas. For ethanol: use 2020-21 onward for the 45 cane districts, and treat the
   other 30 as "no cane data" (≈ small), not zero; cross-check with the Lok Sabha
   Q 324 Annex I yields (75 districts). → DQ-007.
7. **Maize** is reported by fewer districts than the other focus crops (§6): small or
   zero maize in parts of the east. Fine for a volume variable (zero is a value), but note
   it for grain ethanol.
8. **Licence**: ODC-By 1.0 on the mirror (attribution required, commercial use allowed).
   The underlying DES licence isn't stated on the mirror.